# **BAŞLIK VE AÇIKLAMA**

# Project Task 2: Implementation of ML Algorithms from Scratch

## Objective
The goal of this notebook is to satisfy **Task 2** requirements of the project.
We will implement four machine learning algorithms (**Linear Regression, Ridge Regression, K-Nearest Neighbors, and Decision Tree Regressor**) using only raw mathematics and the `numpy` library.

**Note:** External libraries (Scikit-learn) are used **SOLELY** for verification purposes to prove the correctness of our "from-scratch" implementations.

# **KÜTÜPHANELER**

In [14]:
import numpy as np
import pandas as pd
import os
import kagglehub

# Libraries for VERIFICATION ONLY
# We will compare our results with these to prove our code is correct.
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

print("Libraries imported successfully.")

Libraries imported successfully.


# **VERİ YÜKLEME**
## 1. Data Acquisition
We are using the **Student Sleep Patterns** dataset. The code below automatically downloads the dataset and locates the CSV file within the directory.

In [15]:
print("--- Downloading/Checking Datasets ---")
try:
    path_student_sleep = kagglehub.dataset_download('arsalanjamal002/student-sleep-patterns')
except Exception as e:
    print("Error downloading data:", e)

# Helper function to find CSV in nested folders
def find_csv_path(folder_path):
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".csv"):
                return os.path.join(root, file)
    return None

# Load the dataset
csv_path = find_csv_path(path_student_sleep)
if csv_path:
    print(f"Dataset found at: {csv_path}")
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError("CSV file not found.")

print(f"Data Shape: {df.shape}")
df.head()

--- Downloading/Checking Datasets ---
Using Colab cache for faster access to the 'student-sleep-patterns' dataset.
Dataset found at: /kaggle/input/student-sleep-patterns/student_sleep_patterns.csv
Data Shape: (500, 14)


,Student_ID,Age,Gender,University_Year,Sleep_Duration,Study_Hours,Screen_Time,Caffeine_Intake,Physical_Activity,Sleep_Quality,Weekday_Sleep_Start,Weekend_Sleep_Start,Weekday_Sleep_End,Weekend_Sleep_End
0,1,24,Other,2nd Year,7.7,7.9,3.4,2,37,10,14.16,4.05,7.41,7.06
1,2,21,Male,1st Year,6.3,6.0,1.9,5,74,2,8.73,7.10,8.21,10.21
2,3,22,Male,4th Year,5.1,6.7,3.9,5,53,5,20.00,20.47,6.88,10.92
3,4,24,Other,4th Year,6.3,8.6,2.8,4,55,9,19.82,4.08,6.69,9.42
4,5,20,Male,4th Year,4.7,2.7,2.7,0,85,3,20.98,6.12,8.98,9.01


# **VERİ ÖNİŞLEME**
## 2. Data Preprocessing
Since we are implementing algorithms using matrix multiplication (`numpy`), all data must be numerical.
1.  **One-Hot Encoding:** Convert categorical variables (e.g., Gender) to numbers.
2.  **Float Conversion:** Ensure all data types are `float` to avoid `ufunc` errors during matrix inversion.
3.  **Feature Selection:** Remove `Student_ID` (irrelevant) and separate the target `Sleep_Quality`.

In [16]:
# 1. One-Hot Encoding
df_encoded = pd.get_dummies(df, drop_first=True)

# 2. Separate Features (X) and Target (y)
# Drop 'Student_ID' (Identifier) and 'Sleep_Quality' (Target)
X = df_encoded.drop(columns=['Student_ID', 'Sleep_Quality'])
y = df_encoded['Sleep_Quality']

# 3. CRITICAL STEP: Force convert to float
# This prevents numpy type errors during linear algebra operations
X = X.astype(float)
y = y.astype(float)

print("Preprocessing complete.")
print(f"Features (X) Shape: {X.shape}")
print(f"Target (y) Shape: {y.shape}")

Preprocessing complete.
Features (X) Shape: (500, 15)
Target (y) Shape: (500,)


# **YARDIMCI FONKSİYONLAR**
## 3. Helper Functions (Implemented from Scratch)
We implement `train_test_split` and `Mean Squared Error` manually without using sklearn.

In [17]:
def my_train_test_split(X, y, test_size=0.2, random_state=42):
    """
    Splits data into training and testing sets by shuffling indices manually.
    """
    np.random.seed(random_state)
    indices = np.random.permutation(len(X))
    test_samples = int(len(X) * test_size)

    test_indices = indices[:test_samples]
    train_indices = indices[test_samples:]

    return X.iloc[train_indices], X.iloc[test_indices], y.iloc[train_indices], y.iloc[test_indices]

def my_mean_squared_error(y_true, y_pred):
    """
    Calculates the average of squared differences between true and predicted values.
    Formula: (1/n) * Σ(y - y_pred)^2
    """
    return np.mean((y_true - y_pred) ** 2)

# **Algoritma Uygulamaları**

## 4. Algorithm Implementation (From Scratch)

We reimplement the following algorithms using `numpy`:

1.  **Linear Regression:** Using Normal Equation $\theta = (X^T X)^{-1} X^T y$.
2.  **Ridge Regression:** Linear Regression with L2 Regularization $\theta = (X^T X + \alpha I)^{-1} X^T y$.
3.  **K-Nearest Neighbors:** Using Euclidean Distance.
4.  **Decision Tree Regressor:** Using recursive splitting and variance reduction.

In [18]:
# --- MODEL 1: LINEAR REGRESYON (Normal Equation) ---
class MyLinearRegression:
    def __init__(self):
        self.weights = None

    def fit(self, X, y):
        X_vals = X.values if hasattr(X, 'values') else X
        y_vals = y.values if hasattr(y, 'values') else y

        # Add Bias term (column of ones)
        ones = np.ones((len(X_vals), 1))
        X_b = np.c_[ones, X_vals]

        # Calculate weights using Pseudo-Inverse (pinv handles singular matrices better)
        self.weights = np.linalg.pinv(X_b.T.dot(X_b)).dot(X_b.T).dot(y_vals)

    def predict(self, X):
        X_vals = X.values if hasattr(X, 'values') else X
        ones = np.ones((len(X_vals), 1))
        X_b = np.c_[ones, X_vals]
        return X_b.dot(self.weights)

# --- MODEL 2: RIDGE REGRESYON ---
class MyRidgeRegression:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.weights = None

    def fit(self, X, y):
        X_vals = X.values if hasattr(X, 'values') else X
        y_vals = y.values if hasattr(y, 'values') else y

        ones = np.ones((len(X_vals), 1))
        X_b = np.c_[ones, X_vals]

        # Identity Matrix with 0 for bias term
        I = np.eye(X_b.shape[1])
        I[0][0] = 0

        # Ridge Closed-Form Solution
        self.weights = np.linalg.inv(X_b.T.dot(X_b) + self.alpha * I).dot(X_b.T).dot(y_vals)

    def predict(self, X):
        X_vals = X.values if hasattr(X, 'values') else X
        ones = np.ones((len(X_vals), 1))
        X_b = np.c_[ones, X_vals]
        return X_b.dot(self.weights)

# --- MODEL 3: KNN REGRESSION ---
class MyKNNRegressor:
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        # KNN stores training data in memory
        self.X_train = X.values if hasattr(X, 'values') else X
        self.y_train = y.values if hasattr(y, 'values') else y

    def predict(self, X_test):
        X_test_vals = X_test.values if hasattr(X_test, 'values') else X_test
        predictions = []
        for row in X_test_vals:
            # Euclidean Distance
            distances = np.sqrt(np.sum((self.X_train - row)**2, axis=1))
            # Find nearest k indices
            k_indices = np.argsort(distances)[:self.k]
            # Average the target values
            k_nearest_vals = self.y_train[k_indices]
            predictions.append(np.mean(k_nearest_vals))
        return np.array(predictions)

# --- MODEL 4: DECISION TREE REGRESSOR ---
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

class MyDecisionTreeRegressor:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        X_vals = X.values if hasattr(X, 'values') else X
        y_vals = y.values if hasattr(y, 'values') else y
        self.root = self._grow_tree(X_vals, y_vals)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # Stopping criteria
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = np.mean(y)
            return Node(value=leaf_value)

        best_feat, best_thresh = self._best_criteria(X, y, n_features)

        if best_feat is None:
            return Node(value=np.mean(y))

        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)

        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)

    def _best_criteria(self, X, y, n_features):
        best_gain = -1
        split_idx, split_thresh = None, None

        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for threshold in thresholds:
                gain = self._variance_reduction(y, X_column, threshold)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = threshold

        return split_idx, split_thresh

    def _variance_reduction(self, y, X_column, threshold):
        parent_var = np.var(y)
        left_idxs, right_idxs = self._split(X_column, threshold)

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = np.var(y[left_idxs]), np.var(y[right_idxs])

        child_var = (n_l / n) * e_l + (n_r / n) * e_r
        return parent_var - child_var

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def predict(self, X):
        X_vals = X.values if hasattr(X, 'values') else X
        return np.array([self._traverse_tree(x, self.root) for x in X_vals])

    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

# **DENEYLER VE DOĞRULAMA**

## 5. Experiments & Verification
In this section, we run our custom implementations on the test set and compare the **Mean Squared Error (MSE)** with standard Scikit-learn implementations.

* **Linear & Ridge & KNN:** We expect near-exact matches.
* **Decision Tree:** We expect minor deviations due to different splitting/optimization strategies in the library.

In [19]:
# Split Data
X_train, X_test, y_train, y_test = my_train_test_split(X, y, test_size=0.2, random_state=42)

print("="*60)
print("TASK 2: REIMPLEMENTATION & VERIFICATION EXPERIMENTS")
print("="*60)

# --- EXPERIMENT 1: LINEAR REGRESSION ---
print("\n>>> EXP 1: LINEAR REGRESSION <<<")
# Custom
my_lr = MyLinearRegression()
my_lr.fit(X_train, y_train)
mse_my_lr = my_mean_squared_error(y_test, my_lr.predict(X_test))
print(f"1. [My Implementation] MSE: {mse_my_lr:.5f}")

# Sklearn (Verification)
sk_lr = LinearRegression()
sk_lr.fit(X_train, y_train)
mse_sk_lr = my_mean_squared_error(y_test, sk_lr.predict(X_test))
print(f"2. [Sklearn Library]   MSE: {mse_sk_lr:.5f}")

if abs(mse_my_lr - mse_sk_lr) < 1e-4:
    print("RESULT: VERIFIED ✅")
else:
    print("RESULT: DIFFERENT ❌")

# --- EXPERIMENT 2: RIDGE REGRESSION ---
print("\n>>> EXP 2: RIDGE REGRESSION (Alpha=1.0) <<<")
# Custom
my_ridge = MyRidgeRegression(alpha=1.0)
my_ridge.fit(X_train, y_train)
mse_my_ridge = my_mean_squared_error(y_test, my_ridge.predict(X_test))
print(f"1. [My Implementation] MSE: {mse_my_ridge:.5f}")

# Sklearn (Verification)
sk_ridge = Ridge(alpha=1.0)
sk_ridge.fit(X_train, y_train)
mse_sk_ridge = my_mean_squared_error(y_test, sk_ridge.predict(X_test))
print(f"2. [Sklearn Library]   MSE: {mse_sk_ridge:.5f}")

if abs(mse_my_ridge - mse_sk_ridge) < 1e-4:
    print("RESULT: VERIFIED ✅")
else:
    print("RESULT: DIFFERENT ❌")

# --- EXPERIMENT 3: KNN REGRESSION ---
print("\n>>> EXP 3: KNN REGRESSION (k=5) <<<")
# Custom
my_knn = MyKNNRegressor(k=5)
my_knn.fit(X_train, y_train)
mse_my_knn = my_mean_squared_error(y_test, my_knn.predict(X_test))
print(f"1. [My Implementation] MSE: {mse_my_knn:.5f}")

# Sklearn (Verification)
sk_knn = KNeighborsRegressor(n_neighbors=5)
sk_knn.fit(X_train, y_train)
mse_sk_knn = my_mean_squared_error(y_test, sk_knn.predict(X_test))
print(f"2. [Sklearn Library]   MSE: {mse_sk_knn:.5f}")

if abs(mse_my_knn - mse_sk_knn) < 1e-4:
    print("RESULT: VERIFIED ✅")
else:
    print("RESULT: DIFFERENT ❌")

# --- EXPERIMENT 4: DECISION TREE (Max Depth=3) ---
print("\n>>> EXP 4: DECISION TREE (Max Depth=3) <<<")
# Custom
my_tree = MyDecisionTreeRegressor(max_depth=3)
my_tree.fit(X_train, y_train)
mse_my_tree = my_mean_squared_error(y_test, my_tree.predict(X_test))
print(f"1. [My Implementation] MSE: {mse_my_tree:.5f}")

# Sklearn (Verification)
sk_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
sk_tree.fit(X_train, y_train)
mse_sk_tree = my_mean_squared_error(y_test, sk_tree.predict(X_test))
print(f"2. [Sklearn Library]   MSE: {mse_sk_tree:.5f}")

# Calculating Difference
diff = abs(mse_my_tree - mse_sk_tree)
print(f"Difference: {diff:.5f}")

# Accepting slight deviation due to algorithmic differences (e.g., threshold selection)
tolerance = mse_sk_tree * 0.05
if diff < tolerance:
    print("RESULT: VERIFIED ✅ (Acceptable deviation due to algorithmic differences)")
else:
    print("RESULT: DIFFERENT ❌")

print("="*60)

TASK 2: REIMPLEMENTATION & VERIFICATION EXPERIMENTS

>>> EXP 1: LINEAR REGRESSION <<<
1. [My Implementation] MSE: 9.96390
2. [Sklearn Library]   MSE: 9.96390
RESULT: VERIFIED ✅

>>> EXP 2: RIDGE REGRESSION (Alpha=1.0) <<<
1. [My Implementation] MSE: 9.96258
2. [Sklearn Library]   MSE: 9.96258
RESULT: VERIFIED ✅

>>> EXP 3: KNN REGRESSION (k=5) <<<
1. [My Implementation] MSE: 12.13040
2. [Sklearn Library]   MSE: 12.13040
RESULT: VERIFIED ✅

>>> EXP 4: DECISION TREE (Max Depth=3) <<<
1. [My Implementation] MSE: 11.79306
2. [Sklearn Library]   MSE: 11.99101
Difference: 0.19795
RESULT: VERIFIED ✅ (Acceptable deviation due to algorithmic differences)


## 6. Experiments on Second Dataset (Adult Sleep Health)
To satisfy the project requirement ("Experiments will be performed on at least two datasets"), we will now apply our reimplemented algorithms on the **Sleep Health and Lifestyle Dataset**.

* **Target Variable:** `Quality of Sleep` (Numerical, Scale 1-10)
* **Task:** Regression (Predicting sleep quality based on lifestyle factors like Stress, BMI, Activity Level).

In [22]:
print("="*60)
print("DATASET 2: ADULT SLEEP HEALTH & LIFESTYLE")
print("="*60)

# 1. VERİYİ BUL VE YÜKLE
csv_path_adult = find_csv_path(path_sleep_health)
if csv_path_adult:
    print(f"Dataset 2 Loaded: {csv_path_adult}")
    df_adult = pd.read_csv(csv_path_adult)
else:
    raise FileNotFoundError("Second dataset CSV not found.")

# 2. ÖN İŞLEME (PREPROCESSING)
# 'Blood Pressure' sütununu sayısal verilere (Systolic ve Diastolic) çevir
if 'Blood Pressure' in df_adult.columns:
    df_adult[['Systolic_BP', 'Diastolic_BP']] = df_adult['Blood Pressure'].str.split('/', expand=True).astype(float)
    df_adult = df_adult.drop(columns=['Blood Pressure'])

# Kategorik verileri One-Hot Encoding yap
df_adult_encoded = pd.get_dummies(df_adult, drop_first=True)

# Hedef ve Özellikleri Ayır
# Hedef: 'Quality of Sleep'
X_adult = df_adult_encoded.drop(columns=['Person ID', 'Quality of Sleep'])
y_adult = df_adult_encoded['Quality of Sleep']

# FLOAT DÖNÜŞÜMÜ (Hata almamak için)
X_adult = X_adult.astype(float)
y_adult = y_adult.astype(float)

print(f"Features Shape: {X_adult.shape}")
print(f"Target Shape: {y_adult.shape}")

# Veriyi Bölme (Kendi fonksiyonumuzla)
X_train_a, X_test_a, y_train_a, y_test_a = my_train_test_split(X_adult, y_adult, test_size=0.2, random_state=42)

# --- DENEYLER (4 MODELİ DE KULLANIYORUZ) ---

# 1. LINEAR REGRESSION
print("\n>>> EXP 1: LINEAR REGRESSION (Adult Data) <<<")
my_lr_a = MyLinearRegression()
my_lr_a.fit(X_train_a, y_train_a)
mse_my = my_mean_squared_error(y_test_a, my_lr_a.predict(X_test_a))

sk_lr_a = LinearRegression()
sk_lr_a.fit(X_train_a, y_train_a)
mse_sk = my_mean_squared_error(y_test_a, sk_lr_a.predict(X_test_a))

print(f"My Implementation MSE: {mse_my:.5f}")
print(f"Sklearn Library   MSE: {mse_sk:.5f}")
if abs(mse_my - mse_sk) < 1e-4: print("RESULT: VERIFIED ✅")
else: print("RESULT: DIFFERENT ❌")

# 2. RIDGE REGRESSION (Bunu Ekledim)
print("\n>>> EXP 2: RIDGE REGRESSION (Alpha=1.0) (Adult Data) <<<")
my_ridge_a = MyRidgeRegression(alpha=1.0)
my_ridge_a.fit(X_train_a, y_train_a)
mse_my = my_mean_squared_error(y_test_a, my_ridge_a.predict(X_test_a))

sk_ridge_a = Ridge(alpha=1.0)
sk_ridge_a.fit(X_train_a, y_train_a)
mse_sk = my_mean_squared_error(y_test_a, sk_ridge_a.predict(X_test_a))

print(f"My Implementation MSE: {mse_my:.5f}")
print(f"Sklearn Library   MSE: {mse_sk:.5f}")
if abs(mse_my - mse_sk) < 1e-4: print("RESULT: VERIFIED ✅")
else: print("RESULT: DIFFERENT ❌")

# 3. KNN REGRESSION
print("\n>>> EXP 3: KNN REGRESSION (k=5) (Adult Data) <<<")
my_knn_a = MyKNNRegressor(k=5)
my_knn_a.fit(X_train_a, y_train_a)
mse_my = my_mean_squared_error(y_test_a, my_knn_a.predict(X_test_a))

sk_knn_a = KNeighborsRegressor(n_neighbors=5)
sk_knn_a.fit(X_train_a, y_train_a)
mse_sk = my_mean_squared_error(y_test_a, sk_knn_a.predict(X_test_a))

print(f"My Implementation MSE: {mse_my:.5f}")
print(f"Sklearn Library   MSE: {mse_sk:.5f}")
if abs(mse_my - mse_sk) < 1e-4: print("RESULT: VERIFIED ✅")
else: print("RESULT: DIFFERENT ❌")

# 4. DECISION TREE
print("\n>>> EXP 4: DECISION TREE (Adult Data) <<<")
my_tree_a = MyDecisionTreeRegressor(max_depth=3)
my_tree_a.fit(X_train_a, y_train_a)
mse_my = my_mean_squared_error(y_test_a, my_tree_a.predict(X_test_a))

sk_tree_a = DecisionTreeRegressor(max_depth=3, random_state=42)
sk_tree_a.fit(X_train_a, y_train_a)
mse_sk = my_mean_squared_error(y_test_a, sk_tree_a.predict(X_test_a))

diff = abs(mse_my - mse_sk)
print(f"Difference: {diff:.5f}")
if diff < (mse_sk * 0.05): print("RESULT: VERIFIED ✅ (Acceptable deviation)")
else: print("RESULT: DIFFERENT ❌")

print("="*60)

DATASET 2: ADULT SLEEP HEALTH & LIFESTYLE
Dataset 2 Loaded: /kaggle/input/sleep-health-and-lifestyle-dataset/Sleep_health_and_lifestyle_dataset.csv
Features Shape: (374, 23)
Target Shape: (374,)

>>> EXP 1: LINEAR REGRESSION (Adult Data) <<<
My Implementation MSE: 0.05221
Sklearn Library   MSE: 0.05221
RESULT: VERIFIED ✅

>>> EXP 2: RIDGE REGRESSION (Alpha=1.0) (Adult Data) <<<
My Implementation MSE: 0.04979
Sklearn Library   MSE: 0.04979
RESULT: VERIFIED ✅

>>> EXP 3: KNN REGRESSION (k=5) (Adult Data) <<<
My Implementation MSE: 0.19838
Sklearn Library   MSE: 0.19838
RESULT: VERIFIED ✅

>>> EXP 4: DECISION TREE (Adult Data) <<<
Difference: 0.00000
RESULT: VERIFIED ✅ (Acceptable deviation)


# **tek kodda ikisi aynı anda**

In [23]:
# =============================================================================
# UNIFIED EXPERIMENT PIPELINE (TEK SEFERDE İKİ VERİ SETİ)
# =============================================================================

import pandas as pd
import numpy as np
import kagglehub
import os
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

# --- 1. OTOMATİK İŞLEM FONKSİYONU (Engine) ---
def run_full_experiment(X, y, dataset_name):
    """
    Bu fonksiyon verilen X ve y verilerini alır:
    1. Eğitim/Test olarak böler.
    2. 4 Modeli de (Bizimki ve Sklearn) çalıştırır.
    3. Sonuçları bir liste olarak döndürür.
    """
    # Veriyi Böl
    X_train, X_test, y_train, y_test = my_train_test_split(X, y, test_size=0.2, random_state=42)

    results = []

    # Modeller Listesi
    models = [
        ("Linear Regression", MyLinearRegression(), LinearRegression()),
        ("Ridge Regression", MyRidgeRegression(alpha=1.0), Ridge(alpha=1.0)),
        ("KNN Regressor (k=5)", MyKNNRegressor(k=5), KNeighborsRegressor(n_neighbors=5)),
        ("Decision Tree (Depth=3)", MyDecisionTreeRegressor(max_depth=3), DecisionTreeRegressor(max_depth=3, random_state=42))
    ]

    print(f"\nProcessing: {dataset_name} | Samples: {len(X)} | Features: {X.shape[1]}")
    print("-" * 80)
    print(f"{'Algorithm':<25} | {'My MSE':<12} | {'Sklearn MSE':<12} | {'Status'}")
    print("-" * 80)

    for name, my_model, sk_model in models:
        # 1. Bizim Model
        my_model.fit(X_train, y_train)
        my_pred = my_model.predict(X_test)
        my_mse = my_mean_squared_error(y_test, my_pred)

        # 2. Sklearn Model (Doğrulama)
        sk_model.fit(X_train, y_train)
        sk_pred = sk_model.predict(X_test)
        sk_mse = my_mean_squared_error(y_test, sk_pred)

        # 3. Karşılaştırma
        diff = abs(my_mse - sk_mse)

        # Decision Tree için %5 tolerans, diğerleri için çok düşük tolerans
        if "Decision Tree" in name:
            is_verified = diff < (sk_mse * 0.05)
        else:
            is_verified = diff < 1e-4

        status = "✅ Verified" if is_verified else "❌ Different"

        # Ekrana Yazdır
        print(f"{name:<25} | {my_mse:.5f}      | {sk_mse:.5f}      | {status}")

        # Sonuçları Kaydet (Daha sonra tablo yapmak için)
        results.append({
            "Dataset": dataset_name,
            "Algorithm": name,
            "My Implementation MSE": my_mse,
            "Sklearn Library MSE": sk_mse,
            "Difference": diff,
            "Verified": is_verified
        })

    return results

# --- 2. VERİ YÜKLEME VE HAZIRLIK ---

# Yolları Bul
path_student = kagglehub.dataset_download('arsalanjamal002/student-sleep-patterns')
path_adult = kagglehub.dataset_download('uom190346a/sleep-health-and-lifestyle-dataset')

def get_clean_data(dataset_type):
    if dataset_type == "student":
        csv = find_csv_path(path_student)
        df = pd.read_csv(csv)
        df = pd.get_dummies(df, drop_first=True)
        X = df.drop(columns=['Student_ID', 'Sleep_Quality'])
        y = df['Sleep_Quality']

    elif dataset_type == "adult":
        csv = find_csv_path(path_adult)
        df = pd.read_csv(csv)
        # Blood Pressure düzeltmesi
        if 'Blood Pressure' in df.columns:
            df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure'].str.split('/', expand=True).astype(float)
            df = df.drop(columns=['Blood Pressure'])
        df = pd.get_dummies(df, drop_first=True)
        X = df.drop(columns=['Person ID', 'Quality of Sleep'])
        y = df['Quality of Sleep']

    # Float dönüşümü
    return X.astype(float), y.astype(float)

# --- 3. ANA ÇALIŞTIRMA (MAIN LOOP) ---

all_results = []

# A) Öğrenci Verisi
X_s, y_s = get_clean_data("student")
results_s = run_full_experiment(X_s, y_s, "Student Sleep Patterns")
all_results.extend(results_s)

# B) Yetişkin Verisi
X_a, y_a = get_clean_data("adult")
results_a = run_full_experiment(X_a, y_a, "Adult Sleep Health")
all_results.extend(results_a)

# --- 4. FİNAL RAPOR TABLOSU ---
print("\n" + "="*80)
print("FINAL CONSOLIDATED RESULTS TABLE")
print("="*80)
df_results = pd.DataFrame(all_results)

# Tabloyu daha güzel göstermek için
display(df_results[['Dataset', 'Algorithm', 'My Implementation MSE', 'Verified']])

Using Colab cache for faster access to the 'student-sleep-patterns' dataset.
Using Colab cache for faster access to the 'sleep-health-and-lifestyle-dataset' dataset.

Processing: Student Sleep Patterns | Samples: 500 | Features: 15
--------------------------------------------------------------------------------
Algorithm                 | My MSE       | Sklearn MSE  | Status
--------------------------------------------------------------------------------
Linear Regression         | 9.96390      | 9.96390      | ✅ Verified
Ridge Regression          | 9.96258      | 9.96258      | ✅ Verified
KNN Regressor (k=5)       | 12.13040      | 12.13040      | ✅ Verified
Decision Tree (Depth=3)   | 11.79306      | 11.99101      | ✅ Verified

Processing: Adult Sleep Health | Samples: 374 | Features: 23
--------------------------------------------------------------------------------
Algorithm                 | My MSE       | Sklearn MSE  | Status
-----------------------------------------------------

,Dataset,Algorithm,My Implementation MSE,Verified
0,Student Sleep Patterns,Linear Regression,9.963897,True
1,Student Sleep Patterns,Ridge Regression,9.962583,True
2,Student Sleep Patterns,KNN Regressor (k=5),12.130400,True
3,Student Sleep Patterns,Decision Tree (Depth=3),11.793059,True
4,Adult Sleep Health,Linear Regression,0.052213,True
5,Adult Sleep Health,Ridge Regression,0.049788,True
6,Adult Sleep Health,KNN Regressor (k=5),0.198378,True
7,Adult Sleep Health,Decision Tree (Depth=3),0.079626,True
